# GéoMarketing IDF : J10c — Services aux particuliers en Île-de-France

## Objectifs
- Extraire les observations BPE du domaine A, hors restauration A504.
- Conserver le détail des services pour la cartographie.
- Produire des comptages, densités et parts par commune.
- Enrichir le profil communal J10b.
- Contrôler les jointures et documenter les limites.

## Unité de comptage
Une observation BPE, et non nécessairement un établissement physique unique.

## Limites
- La présence dans la source ne confirme pas une ouverture actuelle.
- Une observation ne correspond pas nécessairement à un local recevant du public.
- Les comptages ne mesurent pas la fréquentation.
- Les familles de services doivent être analysées séparément.

In [1]:
#Import des bilbiothèques

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

print("Pandas :", pd.__version__)

Pandas : 2.2.2


In [2]:
#Définir les fichiers

RACINE = Path(
    r"C:\Users\almou\OneDrive\GeoMarketing_IDF"
)

DOSSIER_INTERIM = RACINE / "data" / "interim" / "j10c"
DOSSIER_PROCESSED = RACINE / "data" / "processed"

FICHIER_BPE = (
    RACINE / "data" / "raw" / "bpe" / "BPE25.csv"
)

FICHIER_PROFIL = (
    DOSSIER_PROCESSED / "profil_communes_idf_j10b.csv"
)

FICHIER_DETAIL = (
    DOSSIER_PROCESSED / "services_particuliers_idf_j10c.csv"
)

FICHIER_COMMUNES = (
    DOSSIER_INTERIM / "services_particuliers_communes.csv"
)

FICHIER_SORTIE = (
    DOSSIER_PROCESSED / "profil_communes_idf_j10c.csv"
)

FICHIER_METADATA_J10B = (
    RACINE / "data" / "interim" / "j10b" / "metadata_j10b.json"
)

for fichier in [FICHIER_BPE, FICHIER_PROFIL]:
    if not fichier.is_file():
        raise FileNotFoundError(f"Fichier introuvable : {fichier}")

DOSSIER_INTERIM.mkdir(parents=True, exist_ok=True)
DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Entrée :", FICHIER_PROFIL)
print("Sortie :", FICHIER_SORTIE)

Entrée : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10b.csv
Sortie : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10c.csv


In [4]:
#Fonction de nettoyage et d'enregistrement

def normaliser_nom_colonne(nom):
    nom = unicodedata.normalize(
        "NFKD", str(nom).strip().upper()
    )
    nom = "".join(
        caractere
        for caractere in nom
        if not unicodedata.combining(caractere)
    )
    return re.sub(r"[^A-Z0-9]+", "_", nom).strip("_")


def nettoyer_texte(serie):
    resultat = serie.astype("string").str.strip()

    manquants = {
        "", "NA", "NAN", "NONE", "NULL",
        "<NA>", "_U", "_Z", "[ND]",
    }

    return resultat.mask(
        resultat.str.upper().isin(manquants)
    )


def normaliser_code_commune(serie):
    resultat = nettoyer_texte(serie).str.upper()
    resultat = resultat.str.replace(r"\.0$", "", regex=True)

    masque = resultat.str.fullmatch(r"\d{1,5}", na=False)

    return resultat.where(
        ~masque,
        resultat.str.zfill(5),
    )


def convertir_nombre(serie):
    return pd.to_numeric(
        nettoyer_texte(serie)
        .str.replace("\u202f", "", regex=False)
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def sauvegarder_csv(table, chemin):
    table.to_csv(
        chemin,
        index=False,
        encoding="utf-8-sig",
    )


def sha256_fichier(chemin):
    empreinte = hashlib.sha256()

    with open(chemin, "rb") as flux:
        for bloc in iter(lambda: flux.read(1024 * 1024), b""):
            empreinte.update(bloc)

    return empreinte.hexdigest() 

In [5]:
#Charger le profil J10b

profil = pd.read_csv(
    FICHIER_PROFIL,
    dtype="string",
    encoding="utf-8-sig",
)

profil.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil.columns
]

if profil.columns.duplicated().any():
    raise ValueError("Colonnes dupliquées dans le profil.")

if "CODGEO" not in profil.columns:
    raise ValueError("La colonne CODGEO est absente.")

profil["CODGEO"] = normaliser_code_commune(profil["CODGEO"])

if not profil["CODGEO"].str.fullmatch(r"\d{5}", na=False).all():
    raise ValueError("Codes communaux manquants ou invalides.")

if not profil["CODGEO"].is_unique:
    raise ValueError("Plusieurs lignes existent pour une commune.")

DEPARTEMENTS_IDF = {
    "75", "77", "78", "91", "92", "93", "94", "95"
}

if not profil["CODGEO"].str[:2].isin(DEPARTEMENTS_IDF).all():
    raise ValueError("Le profil contient des codes hors IDF.")

if "NB_OBS_PERIMETRE_J10B_BPE2025" not in profil.columns:
    raise ValueError(
        "Le fichier ne contient pas le total attendu du J10b."
    )

codes_profil = set(profil["CODGEO"])

print("Communes :", len(profil))
print("Colonnes initiales :", len(profil.columns))
display(profil.head())

Communes : 1266
Colonnes initiales : 446


,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU,DENSITE_RESTAURANTS_10000_HAB,DENSITE_RESTAURATION_RAPIDE_10000_HAB,DENSITE_RESTAURATION_TRAD_10000_HAB,PART_RESTAURATION_RAPIDE_PCT,PART_RESTAURATION_TRADITIONNELLE_PCT,INDICE_DENSITE_RAPIDE_IDF_BASE100,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,...,NB_GARES_STATIONS_LOURDES_10000_HAB,NB_POLES_MULTIMODAUX_10000_HAB,INDICE_LIEUX_TRANSPORT_IDF_BASE100,INDICE_LIGNES_FERREES_IDF_BASE100,NB_LIGNES_TRANSPORT_1000_EMPLOIS,NB_GARES_STATIONS_LOURDES_1000_EMPLOIS,CLASSE_DESSERTE_TRANSPORT,NIVEAU_VIE_MEDIAN_2023,TAUX_PAUVRETE_60_2023,STATUT_NIVEAU_VIE_2023,STATUT_PAUVRETE_2023,MILLESIME_REVENUS,GEOGRAPHIE_REVENUS,SOURCE_REVENUS,REVENU_SOURCE_APPARIEE_2023,INDICE_NIVEAU_VIE_MEDIAN_IDF_BASE100_2023,ECART_TAUX_PAUVRETE_IDF_POINTS_2023,NIVEAU_VIE_DISPONIBLE_2023,PAUVRETE_DISPONIBLE_2023,NB_OBS_HYPER_GRANDS_MAGASINS_BPE2025,NB_OBS_SUPER_MULTI_COMMERCES_BPE2025,NB_OBS_SUPERETTES_BPE2025,NB_OBS_EPICERIES_BPE2025,NB_OBS_BOUCHERIES_BPE2025,NB_OBS_SURGELES_BPE2025,NB_OBS_POISSONNERIES_BPE2025,NB_OBS_BOULANGERIES_PATISSERIES_BPE2025,NB_OBS_FRUITS_LEGUMES_BPE2025,NB_OBS_BOISSONS_BPE2025,NB_OBS_AUTRES_ALIMENTAIRES_BPE2025,NB_OBS_ALIMENTAIRES_B2_BPE2025,NB_OBS_EPICERIES_SUPERETTES_BPE2025,NB_OBS_GRANDES_SURFACES_MIXTES_BPE2025,NB_OBS_PERIMETRE_J10B_BPE2025,NB_CATEGORIES_ALIMENTAIRES_B2_PRESENTES,PART_HYPER_GRANDS_MAGASINS_PERIMETRE_J10B_PCT,PART_SUPER_MULTI_COMMERCES_PERIMETRE_J10B_PCT,PART_SUPERETTES_PERIMETRE_J10B_PCT,PART_EPICERIES_PERIMETRE_J10B_PCT,PART_BOUCHERIES_PERIMETRE_J10B_PCT,PART_SURGELES_PERIMETRE_J10B_PCT,PART_POISSONNERIES_PERIMETRE_J10B_PCT,PART_BOULANGERIES_PATISSERIES_PERIMETRE_J10B_PCT,PART_FRUITS_LEGUMES_PERIMETRE_J10B_PCT,PART_BOISSONS_PERIMETRE_J10B_PCT,PART_AUTRES_ALIMENTAIRES_PERIMETRE_J10B_PCT,DENSITE_OBS_ALIMENTAIRES_B2_10000_HAB,DENSITE_OBS_EPICERIES_SUPERETTES_10000_HAB,DENSITE_OBS_BOULANGERIES_10000_HAB,DENSITE_OBS_GRANDES_SURFACES_MIXTES_10000_HAB
0,75056,Paris,75,11,2249975.0,2190327.0,2113705.0,274280.561508206,513386.75602884,457511.658563684,387099.215350194,302819.494109117,154040.332161959,24566.9822779995,787667.317537046,970898.414592524,481426.8085490755,178607.31443995849,12.98,24.29,21.65,37.26,45.93,22.78,8.45,-76622.0,-3.5,-0.59,33650.0,16.8,1551133.75022,188002.0,73.38459010221389,88.94429449710343,20911,8329,12553,29,0,98.93055085738077,39.40474191053151,59.38860910108081,39.8307111089857,60.03060590120033,200.61215141394868,54155.23178,51427.58154,87388.40827,75287.99165,59522.89891,...,1.2347986119160432,0.8610473079261297,40.39419981645854,50.88744885553059,0.1386027134252865,0.1441247338804771,POLE_MULTIMODAL_LOURD,33650.0,16.8,DISPONIBLE,DISPONIBLE,2023,2026-01-01,INSEE_FILOSOFI_2,1,119.28394186458702,-0.5,1,1,44,715,368,2106,649,125,102,2307,387,715,1065,7824,2474,759,8583,9,0.5126412676220435,8.330420598858208,4.287545147384365,24.536875218455087,7.5614586974251425,1.4563672375626238,1.188395665851101,26.878713736455783,4.508912967493883,8.330420598858208,12.408248864033556,37.015572182494715,11.704566152798048,10.914484282338359,3.590851135801827
1,77001,Achères-la-Forêt,77,11,1232.0,1139.0,1183.0,182.135094788161,159.522993092748,189.869837113436,347.331906368833,204.113784168138,89.442063221765,10.5843212469189,34

In [6]:
#Définir les types de services

FAMILLES = {
    "A1": "PUBLICS",
    "A2": "GENERAUX",
    "A3": "AUTOMOBILE",
    "A4": "BATIMENT",
    "A5": "AUTRES_HORS_RESTAURATION",
}

CODES_PUBLICS = [
    "A104", "A105", "A108", "A109",
    "A120", "A121", "A122",
    "A124", "A125", "A126", "A128", "A129",
    "A130", "A131", "A132", "A133", "A134",
    "A135", "A136", "A137", "A138", "A139", "A140",
]

NOMS_ANALYSE = {
    **{code: f"SERVICE_PUBLIC_{code}" for code in CODES_PUBLICS},

    "A203": "BANQUES",
    "A205": "FUNERAIRE",
    "A206": "POSTE_BUREAUX",
    "A207": "POSTE_RELAIS",
    "A208": "POSTE_AGENCES",

    "A301": "REPARATION_VEHICULES_MATERIEL_AGRICOLE",
    "A302": "CONTROLE_VEHICULES",
    "A303": "LOCATION_VEHICULES",
    "A304": "FORMATION_CONDUITE",

    "A401": "MACONNERIE",
    "A402": "PLATRERIE_PEINTURE",
    "A403": "MENUISERIE_CHARPENTE_SERRURERIE",
    "A404": "PLOMBERIE_COUVERTURE_CHAUFFAGE",
    "A405": "ELECTRICITE",
    "A406": "ENTREPRISES_BATIMENT",

    "A501": "COIFFURE",
    "A502": "VETERINAIRES",
    "A503": "INTERIM",
    "A505": "IMMOBILIER",
    "A506": "ENTRETIEN_LINGE",
    "A507": "BEAUTE_ONGLES",
}

CODES_RETENUS = list(NOMS_ANALYSE)

assert "A504" not in CODES_RETENUS

referentiel = pd.DataFrame({
    "TYPEQU_ANALYSE": CODES_RETENUS,
    "NOM_ANALYSE": [NOMS_ANALYSE[code] for code in CODES_RETENUS],
})

referentiel["FAMILLE_CODE"] = (
    referentiel["TYPEQU_ANALYSE"].str[:2]
)

referentiel["FAMILLE_ANALYSE"] = (
    referentiel["FAMILLE_CODE"].map(FAMILLES)
)

sauvegarder_csv(
    referentiel,
    DOSSIER_INTERIM / "referentiel_services_j10c.csv",
)

display(referentiel)

,TYPEQU_ANALYSE,NOM_ANALYSE,FAMILLE_CODE,FAMILLE_ANALYSE
0,A104,SERVICE_PUBLIC_A104,A1,PUBLICS
1,A105,SERVICE_PUBLIC_A105,A1,PUBLICS
2,A108,SERVICE_PUBLIC_A108,A1,PUBLICS
3,A109,SERVICE_PUBLIC_A109,A1,PUBLICS
4,A120,SERVICE_PUBLIC_A120,A1,PUBLICS
5,A121,SERVICE_PUBLIC_A121,A1,PUBLICS
6,A122,SERVICE_PUBLIC_A122,A1,PUBLICS
7,A124,SERVICE_PUBLIC_A124,A1,PUBLICS
8,A125,SERVICE_PUBLIC_A125,A1,PUBLICS
9,A126,SERVICE_PUBLIC_A126,A1,PUBLICS


In [7]:
#Contrôler le fichier BPE

with open(
    FICHIER_BPE,
    "r",
    encoding="utf-8-sig",
) as flux:
    premiere_ligne = flux.readline()

separateurs_valides = []

for separateur in [";", ",", "\t", "|"]:
    colonnes = next(
        csv.reader([premiere_ligne], delimiter=separateur)
    )

    colonnes = [
        normaliser_nom_colonne(colonne)
        for colonne in colonnes
    ]

    if {"AN", "DEPCOM", "TYPEQU"}.issubset(colonnes):
        separateurs_valides.append(separateur)

if len(separateurs_valides) != 1:
    raise ValueError(
        "En-tête BPE détaillé non reconnu. "
        "Il faut notamment AN, DEPCOM et TYPEQU."
    )

SEPARATEUR_BPE = separateurs_valides[0]

entete = pd.read_csv(
    FICHIER_BPE,
    sep=SEPARATEUR_BPE,
    encoding="utf-8-sig",
    nrows=0,
)

noms = [
    normaliser_nom_colonne(colonne)
    for colonne in entete.columns
]

if len(noms) != len(set(noms)):
    raise ValueError("Noms de colonnes ambigus après normalisation.")

print("Séparateur :", repr(SEPARATEUR_BPE))
print("Nombre de colonnes :", len(noms))

Séparateur : ';'
Nombre de colonnes : 95


In [8]:
#Extraire les services

blocs_retenus = []
nombre_lignes_lues = 0
codes_a_inconnus = set()

with pd.read_csv(
    FICHIER_BPE,
    sep=SEPARATEUR_BPE,
    dtype="string",
    encoding="utf-8-sig",
    keep_default_na=False,
    chunksize=20_000,
) as lecteur:

    for numero_bloc, bloc in enumerate(lecteur, start=1):
        bloc.columns = [
            normaliser_nom_colonne(colonne)
            for colonne in bloc.columns
        ]

        bloc["RANG_SOURCE_J10C"] = np.arange(
            nombre_lignes_lues + 1,
            nombre_lignes_lues + len(bloc) + 1,
        )

        nombre_lignes_lues += len(bloc)

        types = nettoyer_texte(bloc["TYPEQU"]).str.upper()
        communes = normaliser_code_commune(bloc["DEPCOM"])

        est_idf = communes.str[:2].isin(DEPARTEMENTS_IDF)

        est_service_hors_restauration = (
            types.str.startswith("A", na=False)
            & types.ne("A504").fillna(False)
        )

        inconnus = (
            est_idf
            & est_service_hors_restauration
            & ~types.isin(CODES_RETENUS)
        )

        codes_a_inconnus.update(
            types.loc[inconnus].dropna().tolist()
        )

        masque = est_idf & types.isin(CODES_RETENUS)

        selection = bloc.loc[masque].copy()

        if not selection.empty:
            selection["TYPEQU_ANALYSE"] = types.loc[masque]
            selection["CODGEO_SOURCE"] = communes.loc[masque]
            blocs_retenus.append(selection)

        if numero_bloc % 25 == 0:
            print(f"{nombre_lignes_lues:,} lignes examinées")

if codes_a_inconnus:
    raise ValueError(
        "Types de services à vérifier dans la nomenclature : "
        + ", ".join(sorted(codes_a_inconnus))
    )

if not blocs_retenus:
    raise ValueError("Aucune observation retenue.")

services = pd.concat(blocs_retenus, ignore_index=True)
del blocs_retenus

if not nettoyer_texte(services["AN"]).eq("2025").fillna(False).all():
    raise ValueError("L'extraction contient un millésime différent de 2025.")

print("Observations de services retenues :", len(services))

500,000 lignes examinées
1,000,000 lignes examinées
1,500,000 lignes examinées
2,000,000 lignes examinées
2,500,000 lignes examinées
Observations de services retenues : 184760


In [9]:
#Ajouter la tracabilité et les familles

HASH_BPE = sha256_fichier(FICHIER_BPE)
DATE_TRAITEMENT = datetime.now(timezone.utc).isoformat()

services["ID_OBSERVATION"] = (
    "BPE2025_"
    + HASH_BPE[:16]
    + "_"
    + services["RANG_SOURCE_J10C"].astype("string")
)

services["NOM_ANALYSE"] = (
    services["TYPEQU_ANALYSE"].map(NOMS_ANALYSE)
)

services["FAMILLE_CODE"] = (
    services["TYPEQU_ANALYSE"].str[:2]
)

services["FAMILLE_ANALYSE"] = (
    services["FAMILLE_CODE"].map(FAMILLES)
)

services["SOURCE_DONNEES"] = "INSEE_BPE_2025"
services["STATUT_OUVERTURE_ACTUELLE"] = "NON_VERIFIE"
services["ACCUEIL_PUBLIC_VERIFIE"] = "NON_VERIFIE"

assert services["ID_OBSERVATION"].is_unique
assert services["FAMILLE_ANALYSE"].notna().all()

display(services["FAMILLE_ANALYSE"].value_counts())

FAMILLE_ANALYSE
BATIMENT                    100388
AUTRES_HORS_RESTAURATION     56489
AUTOMOBILE                   18930
GENERAUX                      6531
PUBLICS                       2422
Name: count, dtype: int64

In [11]:
#Harmoniser les communes

services["CODGEO"] = services["CODGEO_SOURCE"]

codes_arrondissements_paris = {
    f"751{numero:02d}" for numero in range(1, 21)
}

if codes_profil & codes_arrondissements_paris:
    raise ValueError(
        "Le profil doit être communal, sans arrondissements parisiens."
    )

if "75056" in codes_profil:
    services.loc[
        services["CODGEO"].isin(codes_arrondissements_paris),
        "CODGEO",
    ] = "75056"

if "93066" in codes_profil and "93059" not in codes_profil:
    services.loc[
        services["CODGEO"].eq("93059"),
        "CODGEO",
    ] = "93066"

hors_profil = services.loc[
    ~services["CODGEO"].isin(codes_profil)
].copy()

sauvegarder_csv(
    hors_profil,
    DOSSIER_INTERIM / "services_hors_profil.csv",
)

print("Observations hors profil :", len(hors_profil))

if not hors_profil.empty:
    display(
        hors_profil[
            ["CODGEO_SOURCE", "CODGEO", "TYPEQU_ANALYSE"]
        ].drop_duplicates()
    )
    raise ValueError(
        "Résoudre les communes non appariées avant de poursuivre."
    )

Observations hors profil : 0


In [12]:
#Préparer les noms et adresses

colonnes_facultatives = [
    "SIRET", "NOMRS", "CNOMRS",
    "NUMVOIE", "INDREP", "TYPVOIE", "LIBVOIE", "CADR",
    "CODPOS", "LATITUDE", "LONGITUDE",
]

for colonne in colonnes_facultatives:
    if colonne not in services.columns:
        services[colonne] = pd.Series(
            pd.NA,
            index=services.index,
            dtype="string",
        )

services["NOM_AFFICHAGE"] = (
    nettoyer_texte(services["NOMRS"])
    .fillna(nettoyer_texte(services["CNOMRS"]))
)

parties_adresse = services[
    ["NUMVOIE", "INDREP", "TYPVOIE", "LIBVOIE", "CADR"]
].apply(nettoyer_texte)

services["ADRESSE_RECOMPOSEE"] = (
    parties_adresse.fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .replace("", pd.NA)
)

display(
    services[
        [
            "TYPEQU_ANALYSE",
            "NOM_AFFICHAGE",
            "ADRESSE_RECOMPOSEE",
            "CODGEO",
        ]
    ].head(10)
)

,TYPEQU_ANALYSE,NOM_AFFICHAGE,ADRESSE_RECOMPOSEE,CODGEO
0,A105,COUR D'APPEL DE PARIS,8 BD DU PALAIS,75056
1,A120,DIRECTION REGIONALE FINANCES PUBLIQUES ILE DE FRANCE ET DEPARTEMENT PARIS,6 RUE SAINT HYACINTHE,75056
2,A134,COUR CRIMINELLE DÉPARTEMENTALE DE PARIS,34 QUAI DES ORFEVRES,75056
3,A135,COUR D'ASSISES DE PARIS,34 QUAI DES ORFEVRES,75056
4,A203,CREDIT LYONNAIS,20 AV DE L OPERA,75056
5,A203,BNP PARIBAS,22 AV DE L OPERA,75056
6,A203,BANCO BILBAO VIZCAYA ARGENTARIA,29 AV DE L OPERA MR BENSOUSSAN ARNAUD,75056
7,A203,CREDIT LYONNAIS,47 BD DE SEBASTOPOL,75056
8,A203,CREDIT LYONNAIS,4 PL ANDRE MALRAUX,75056
9,A203,J.P MORGAN CHASE BANK NATIONAL ASSOCIAT,14 PL VENDOME,75056


In [13]:
#Contrôler les répétitions de SIRET

services["SIRET_NETTOYE"] = nettoyer_texte(services["SIRET"])

services["SIRET_FORMAT_CONFORME"] = (
    services["SIRET_NETTOYE"]
    .str.fullmatch(r"\d{14}", na=False)
    & services["SIRET_NETTOYE"].ne("0" * 14).fillna(False)
)

services["SIRET_TYPE_REPETE"] = (
    services["SIRET_FORMAT_CONFORME"]
    & services.duplicated(
        subset=["SIRET_NETTOYE", "TYPEQU_ANALYSE"],
        keep=False,
    )
)

repetitions = services.loc[
    services["SIRET_TYPE_REPETE"]
].copy()

sauvegarder_csv(
    repetitions,
    DOSSIER_INTERIM / "siret_types_repetes.csv",
)

print(
    "Observations avec SIRET et type répétés :",
    len(repetitions),
)

Observations avec SIRET et type répétés : 0


In [14]:
#Contrôler les coordonnées

services["LATITUDE_NUM"] = convertir_nombre(
    services["LATITUDE"]
)

services["LONGITUDE_NUM"] = convertir_nombre(
    services["LONGITUDE"]
)

coordonnees_presentes = (
    services["LATITUDE_NUM"].notna()
    & services["LONGITUDE_NUM"].notna()
)

coordonnees_plausibles = (
    services["LATITUDE_NUM"].between(48.0, 49.3)
    & services["LONGITUDE_NUM"].between(1.4, 3.7)
).fillna(False)

services["STATUT_COORDONNEES"] = "ABSENTES_OU_NON_NUMERIQUES"

services.loc[
    coordonnees_presentes,
    "STATUT_COORDONNEES",
] = "HORS_RECTANGLE_CONTROLE"

services.loc[
    coordonnees_plausibles,
    "STATUT_COORDONNEES",
] = "PLAUSIBLES_A_VERIFIER"

controle_coordonnees = (
    services.groupby(
        ["FAMILLE_ANALYSE", "STATUT_COORDONNEES"]
    )
    .size()
    .reset_index(name="NB_OBSERVATIONS")
)

sauvegarder_csv(
    controle_coordonnees,
    DOSSIER_INTERIM / "controle_coordonnees.csv",
)

display(controle_coordonnees)

,FAMILLE_ANALYSE,STATUT_COORDONNEES,NB_OBSERVATIONS
0,AUTOMOBILE,ABSENTES_OU_NON_NUMERIQUES,1335
1,AUTOMOBILE,PLAUSIBLES_A_VERIFIER,17595
2,AUTRES_HORS_RESTAURATION,ABSENTES_OU_NON_NUMERIQUES,7602
3,AUTRES_HORS_RESTAURATION,PLAUSIBLES_A_VERIFIER,48887
4,BATIMENT,ABSENTES_OU_NON_NUMERIQUES,4149
5,BATIMENT,PLAUSIBLES_A_VERIFIER,96239
6,GENERAUX,ABSENTES_OU_NON_NUMERIQUES,29
7,GENERAUX,PLAUSIBLES_A_VERIFIER,6502
8,PUBLICS,PLAUSIBLES_A_VERIFIER,2422


In [28]:
#Compter par type et commune

COLONNES_TYPES = {
    code: f"NB_OBS_{code}_BPE2025"
    for code in CODES_RETENUS
}

comptages = pd.crosstab(
    services["CODGEO"],
    services["TYPEQU_ANALYSE"],
)

comptages = (
    comptages.reindex(
        columns=CODES_RETENUS,
        fill_value=0,
    )
    .rename(columns=COLONNES_TYPES)
    .reset_index()
)

indicateurs = profil[["CODGEO"]].merge(
    comptages,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

colonnes_types = list(COLONNES_TYPES.values())

indicateurs[colonnes_types] = (
    indicateurs[colonnes_types]
    .fillna(0)
    .astype("int64")
)

display(indicateurs.head())

,CODGEO,NB_OBS_A104_BPE2025,NB_OBS_A105_BPE2025,NB_OBS_A108_BPE2025,NB_OBS_A109_BPE2025,NB_OBS_A120_BPE2025,NB_OBS_A121_BPE2025,NB_OBS_A122_BPE2025,NB_OBS_A124_BPE2025,NB_OBS_A125_BPE2025,NB_OBS_A126_BPE2025,NB_OBS_A128_BPE2025,NB_OBS_A129_BPE2025,NB_OBS_A130_BPE2025,NB_OBS_A131_BPE2025,NB_OBS_A132_BPE2025,NB_OBS_A133_BPE2025,NB_OBS_A134_BPE2025,NB_OBS_A135_BPE2025,NB_OBS_A136_BPE2025,NB_OBS_A137_BPE2025,NB_OBS_A138_BPE2025,NB_OBS_A139_BPE2025,NB_OBS_A140_BPE2025,NB_OBS_A203_BPE2025,NB_OBS_A205_BPE2025,NB_OBS_A206_BPE2025,NB_OBS_A207_BPE2025,NB_OBS_A208_BPE2025,NB_OBS_A301_BPE2025,NB_OBS_A302_BPE2025,NB_OBS_A303_BPE2025,NB_OBS_A304_BPE2025,NB_OBS_A401_BPE2025,NB_OBS_A402_BPE2025,NB_OBS_A403_BPE2025,NB_OBS_A404_BPE2025,NB_OBS_A405_BPE2025,NB_OBS_A406_BPE2025,NB_OBS_A501_BPE2025,NB_OBS_A502_BPE2025,NB_OBS_A503_BPE2025,NB_OBS_A505_BPE2025,NB_OBS_A506_BPE2025,NB_OBS_A507_BPE2025
0,75056,1,1,1,1,23,0,18,3,0,1,12,20,1,1,0,10,1,1,2,1,1,1,19,1265,209,128,38,0,912,26,1068,346,2006,3369,1757,2940,2933,3561,3548,246,320,8136,731,3568
1,77001,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,2,1,1,0,2,0,1,1,0,1,0,2
2,77002,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,1,3,0,5,2,4,4,1,0,0,0,0,0
3,77003,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,1,0,2,1,0,0,0,0,0,0,0,0
4,77004,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,1


In [31]:
CODES_RETENUS

['A104',
 'A105',
 'A108',
 'A109',
 'A120',
 'A121',
 'A122',
 'A124',
 'A125',
 'A126',
 'A128',
 'A129',
 'A130',
 'A131',
 'A132',
 'A133',
 'A134',
 'A135',
 'A136',
 'A137',
 'A138',
 'A139',
 'A140',
 'A203',
 'A205',
 'A206',
 'A207',
 'A208',
 'A301',
 'A302',
 'A303',
 'A304',
 'A401',
 'A402',
 'A403',
 'A404',
 'A405',
 'A406',
 'A501',
 'A502',
 'A503',
 'A505',
 'A506',
 'A507']

In [32]:
#Construire les totaux par famille

COLONNES_FAMILLES = {
    famille: f"NB_OBS_SERVICES_{famille}_BPE2025"
    for famille in FAMILLES
}

for famille, colonne_sortie in COLONNES_FAMILLES.items():
    colonnes_a_sommer = [
        COLONNES_TYPES[code]
        for code in CODES_RETENUS
        if code.startswith(famille)
    ]

    indicateurs[colonne_sortie] = (
        indicateurs[colonnes_a_sommer].sum(axis=1)
    )

COLONNE_TOTAL = "NB_OBS_SERVICES_HORS_RESTAURATION_BPE2025"

indicateurs[COLONNE_TOTAL] = (
    indicateurs[colonnes_types].sum(axis=1)
)

indicateurs["NB_TYPES_SERVICES_PRESENTS_J10C"] = (
    indicateurs[colonnes_types].gt(0).sum(axis=1)
)

assert (
    indicateurs[list(COLONNES_FAMILLES.values())].sum(axis=1)
    == indicateurs[COLONNE_TOTAL]
).all()

assert indicateurs[COLONNE_TOTAL].sum() == len(services)

print("Agrégation par type et famille cohérente.")

Agrégation par type et famille cohérente.


In [33]:
indicateurs

,CODGEO,NB_OBS_A104_BPE2025,NB_OBS_A105_BPE2025,NB_OBS_A108_BPE2025,NB_OBS_A109_BPE2025,NB_OBS_A120_BPE2025,NB_OBS_A121_BPE2025,NB_OBS_A122_BPE2025,NB_OBS_A124_BPE2025,NB_OBS_A125_BPE2025,NB_OBS_A126_BPE2025,NB_OBS_A128_BPE2025,NB_OBS_A129_BPE2025,NB_OBS_A130_BPE2025,NB_OBS_A131_BPE2025,NB_OBS_A132_BPE2025,NB_OBS_A133_BPE2025,NB_OBS_A134_BPE2025,NB_OBS_A135_BPE2025,NB_OBS_A136_BPE2025,NB_OBS_A137_BPE2025,NB_OBS_A138_BPE2025,NB_OBS_A139_BPE2025,NB_OBS_A140_BPE2025,NB_OBS_A203_BPE2025,NB_OBS_A205_BPE2025,NB_OBS_A206_BPE2025,NB_OBS_A207_BPE2025,NB_OBS_A208_BPE2025,NB_OBS_A301_BPE2025,NB_OBS_A302_BPE2025,NB_OBS_A303_BPE2025,NB_OBS_A304_BPE2025,NB_OBS_A401_BPE2025,NB_OBS_A402_BPE2025,NB_OBS_A403_BPE2025,NB_OBS_A404_BPE2025,NB_OBS_A405_BPE2025,NB_OBS_A406_BPE2025,NB_OBS_A501_BPE2025,NB_OBS_A502_BPE2025,NB_OBS_A503_BPE2025,NB_OBS_A505_BPE2025,NB_OBS_A506_BPE2025,NB_OBS_A507_BPE2025,NB_OBS_SERVICES_A1_BPE2025,NB_OBS_SERVICES_A2_BPE2025,NB_OBS_SERVICES_A3_BPE2025,NB_OBS_SERVICES_A4_BPE2025,NB_OBS_SERVICES_A5_BPE2025,NB_OBS_SERVICES_HORS_RESTAURATION_BPE2025,NB_TYPES_SERVICES_PRESENTS_J10C
0,75056,1,1,1,1,23,0,18,3,0,1,12,20,1,1,0,10,1,1,2,1,1,1,19,1265,209,128,38,0,912,26,1068,346,2006,3369,1757,2940,2933,3561,3548,246,320,8136,731,3568,119,1640,2352,16566,16549,37226,40
1,77001,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,2,1,1,0,2,0,1,1,0,1,0,2,1,0,2,6,5,14,10
2,77002,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,1,3,0,5,2,4,4,1,0,0,0,0,0,1,0,3,18,1,23,9
3,77003,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,1,0,2,1,0,0,0,0,0,0,0,0,2,0,2,4,0,8,7
4,77004,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,1,1,0,0,3,1,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1261,95676,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,2,0,1,1,0,0,3,3,7,6
1262,95678,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,2,0,0,0,5,1,3,2,0,1,2,0,0,1,0,0,1,1,2,12,3,19,10
1263,95680,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,2,2,1,0,0,28,2,8,18,57,57,25,46,50,48,27,1,0,21,6,18,4,5,56,283,73,421,22
1264,95682,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,2,0,0,3,0,0,0,1,0,0,1,0,1,7,1,10,7


In [34]:
#Calculer les parts en pourcentage

total_services = indicateurs[COLONNE_TOTAL].replace(0, np.nan)

PARTS_TYPES = {}

for code, colonne_nombre in COLONNES_TYPES.items():
    colonne_part = f"PART_{code}_SERVICES_J10C_PCT"

    indicateurs[colonne_part] = (
        indicateurs[colonne_nombre]
        .div(total_services)
        .mul(100)
    )

    PARTS_TYPES[code] = colonne_part

PARTS_FAMILLES = {}

for famille, colonne_nombre in COLONNES_FAMILLES.items():
    colonne_part = f"PART_{famille}_SERVICES_J10C_PCT"

    indicateurs[colonne_part] = (
        indicateurs[colonne_nombre]
        .div(total_services)
        .mul(100)
    )

    PARTS_FAMILLES[famille] = colonne_part

avec_services = indicateurs[COLONNE_TOTAL].gt(0)

for colonnes_parts in [
    list(PARTS_TYPES.values()),
    list(PARTS_FAMILLES.values()),
]:
    valeurs = indicateurs.loc[avec_services, colonnes_parts]

    assert valeurs.notna().all().all()
    assert ((valeurs >= 0) & (valeurs <= 100)).all().all()

    assert np.allclose(
        valeurs.sum(axis=1).to_numpy(dtype=float),
        100.0,
    )

    assert indicateurs.loc[
        ~avec_services, colonnes_parts
    ].isna().all().all()

print("Parts contrôlées.")

Parts contrôlées.


In [35]:
#Utiliser j10b

# Renseigner explicitement si nécessaire.
COLONNE_POPULATION = None

metadata_j10b = {}

if FICHIER_METADATA_J10B.is_file():
    with open(
        FICHIER_METADATA_J10B,
        "r",
        encoding="utf-8",
    ) as flux:
        metadata_j10b = json.load(flux)

if COLONNE_POPULATION is None:
    COLONNE_POPULATION = metadata_j10b.get("colonne_population")

if COLONNE_POPULATION is None:
    candidats = [
        colonne
        for colonne in [
            "POPULATION",
            "POPULATION_MUNICIPALE",
            "P23_POP",
            "P22_POP",
            "POP",
        ]
        if colonne in profil.columns
    ]

    if len(candidats) == 1:
        COLONNE_POPULATION = candidats[0]
    else:
        print("Candidats :", candidats)
        raise ValueError(
            "Renseigne COLONNE_POPULATION avec la colonne "
            "utilisée au J10b."
        )

if COLONNE_POPULATION not in profil.columns:
    raise ValueError(
        f"Colonne introuvable : {COLONNE_POPULATION}"
    )

# Alignement explicite sur les codes communaux.
population_par_code = pd.Series(
    convertir_nombre(profil[COLONNE_POPULATION]).to_numpy(),
    index=profil["CODGEO"],
)

population = pd.to_numeric(
    indicateurs["CODGEO"].map(population_par_code),
    errors="coerce",
)

population_valide = population.gt(0).fillna(False)
denominateur = population.where(population_valide)

print("Population utilisée :", COLONNE_POPULATION)
print(
    "Communes sans population positive exploitable :",
    int((~population_valide).sum()),
)

Population utilisée : POPULATION_2022
Communes sans population positive exploitable : 0


In [36]:
#Calcul des densités

DENSITES = {}

for code, colonne_nombre in COLONNES_TYPES.items():
    DENSITES[colonne_nombre] = (
        f"DENSITE_OBS_{code}_10000_HAB"
    )

for famille, colonne_nombre in COLONNES_FAMILLES.items():
    DENSITES[colonne_nombre] = (
        f"DENSITE_OBS_SERVICES_{famille}_10000_HAB"
    )

DENSITES[COLONNE_TOTAL] = (
    "DENSITE_OBS_SERVICES_HORS_RESTAURATION_10000_HAB"
)

for colonne_nombre, colonne_densite in DENSITES.items():
    indicateurs[colonne_densite] = (
        indicateurs[colonne_nombre]
        .div(denominateur)
        .mul(10_000)
    )

print("Densités calculées :", len(DENSITES))

Densités calculées : 50


C:\Users\almou\AppData\Local\Temp\ipykernel_33392\1590423741.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  indicateurs[colonne_densite] = (
C:\Users\almou\AppData\Local\Temp\ipykernel_33392\1590423741.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  indicateurs[colonne_densite] = (
C:\Users\almou\AppData\Local\Temp\ipykernel_33392\1590423741.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all colu

In [37]:
#Joindre les indicateurs au profil

nouvelles_colonnes = set(indicateurs.columns) - {"CODGEO"}
collisions = nouvelles_colonnes & set(profil.columns)

if collisions:
    raise ValueError(
        "Colonnes déjà présentes dans le profil d'entrée : "
        + ", ".join(sorted(collisions))
    )

profil_j10c = profil.merge(
    indicateurs,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

assert len(profil_j10c) == len(profil)
assert profil_j10c["CODGEO"].is_unique

pd.testing.assert_frame_equal(
    profil.sort_values("CODGEO").reset_index(drop=True),
    profil_j10c[profil.columns]
    .sort_values("CODGEO")
    .reset_index(drop=True),
)

print("Colonnes initiales :", len(profil.columns))
print("Colonnes après enrichissement :", len(profil_j10c.columns))
print("Données antérieures conservées.")

Colonnes initiales : 446
Colonnes après enrichissement : 596
Données antérieures conservées.


In [38]:
#Examiner les communes et catégories

colonne_nom = next(
    (
        colonne
        for colonne in ["LIBELLE", "NOM_COMMUNE", "LIBGEO", "NOM_COM"]
        if colonne in profil_j10c.columns
    ),
    None,
)

colonnes_affichage = ["CODGEO"]

if colonne_nom:
    colonnes_affichage.append(colonne_nom)

colonnes_affichage += [
    COLONNE_POPULATION,
    COLONNE_TOTAL,
    *COLONNES_FAMILLES.values(),
]

display(
    profil_j10c.sort_values(
        COLONNE_TOTAL,
        ascending=False,
    )[colonnes_affichage].head(20)
)

print("Cergy et Pontoise :")

display(
    profil_j10c.loc[
        profil_j10c["CODGEO"].isin(["95127", "95500"]),
        colonnes_affichage,
    ]
)

bilan_types = (
    services.groupby(
        ["TYPEQU_ANALYSE", "NOM_ANALYSE", "FAMILLE_ANALYSE"]
    )
    .size()
    .reset_index(name="NB_OBSERVATIONS")
    .sort_values("NB_OBSERVATIONS", ascending=False)
)

sauvegarder_csv(
    bilan_types,
    DOSSIER_INTERIM / "bilan_types_services.csv",
)

display(bilan_types)

,CODGEO,NOM_COMMUNE,POPULATION_2022,NB_OBS_SERVICES_HORS_RESTAURATION_BPE2025,NB_OBS_SERVICES_A1_BPE2025,NB_OBS_SERVICES_A2_BPE2025,NB_OBS_SERVICES_A3_BPE2025,NB_OBS_SERVICES_A4_BPE2025,NB_OBS_SERVICES_A5_BPE2025
0,75056,Paris,2113705.0,37226,119,1640,2352,16566,16549
1027,93066,Saint-Denis,148907.0,2219,17,55,275,1503,369
1088,95018,Argenteuil,107135.0,1987,11,36,267,1337,336
1007,93029,Drancy,71312.0,1874,7,27,162,1434,244
1016,93048,Montreuil,110758.0,1818,9,46,185,1259,319
998,93005,Aulnay-sous-Bois,86360.0,1624,8,27,199,1160,230
965,92012,Boulogne-Billancourt,120205.0,1572,8,66,104,592,802
1070,94068,Saint-Maur-des-Fossés,76010.0,1396,8,46,94,755,493
1043,94017,Champigny-sur-Marne,78367.0,1380,7,28,137,950,258
997,93001,Aubervilliers,89489.0,1318,9,23,125,937,224


Cergy et Pontoise :


,CODGEO,NOM_COMMUNE,POPULATION_2022,NB_OBS_SERVICES_HORS_RESTAURATION_BPE2025,NB_OBS_SERVICES_A1_BPE2025,NB_OBS_SERVICES_A2_BPE2025,NB_OBS_SERVICES_A3_BPE2025,NB_OBS_SERVICES_A4_BPE2025,NB_OBS_SERVICES_A5_BPE2025
1119,95127,Cergy,69578.0,718,13,36,105,361,203
1220,95500,Pontoise,31623.0,483,14,23,48,250,148


,TYPEQU_ANALYSE,NOM_ANALYSE,FAMILLE_ANALYSE,NB_OBSERVATIONS
41,A505,IMMOBILIER,AUTRES_HORS_RESTAURATION,23813
35,A404,PLOMBERIE_COUVERTURE_CHAUFFAGE,BATIMENT,21658
33,A402,PLATRERIE_PEINTURE,BATIMENT,18142
36,A405,ELECTRICITE,BATIMENT,17647
37,A406,ENTREPRISES_BATIMENT,BATIMENT,16287
32,A401,MACONNERIE,BATIMENT,15210
43,A507,BEAUTE_ONGLES,AUTRES_HORS_RESTAURATION,14123
38,A501,COIFFURE,AUTRES_HORS_RESTAURATION,13637
34,A403,MENUISERIE_CHARPENTE_SERRURERIE,BATIMENT,11444
28,A301,REPARATION_VEHICULES_MATERIEL_AGRICOLE,AUTOMOBILE,10629


In [39]:
lignes_dictionnaire = []

for code in CODES_RETENUS:
    lignes_dictionnaire.extend([
        {
            "COLONNE": COLONNES_TYPES[code],
            "DESCRIPTION": f"Observations BPE : {NOMS_ANALYSE[code]}",
            "UNITE": "observations",
            "DENOMINATEUR": "",
        },
        {
            "COLONNE": PARTS_TYPES[code],
            "DESCRIPTION": f"Part du type {code} dans les services J10c",
            "UNITE": "%",
            "DENOMINATEUR": COLONNE_TOTAL,
        },
        {
            "COLONNE": DENSITES[COLONNES_TYPES[code]],
            "DESCRIPTION": f"Densité du type {code}",
            "UNITE": "observations pour 10 000 habitants",
            "DENOMINATEUR": COLONNE_POPULATION,
        },
    ])

for famille, colonne_nombre in COLONNES_FAMILLES.items():
    lignes_dictionnaire.extend([
        {
            "COLONNE": colonne_nombre,
            "DESCRIPTION": f"Observations de la famille {FAMILLES[famille]}",
            "UNITE": "observations",
            "DENOMINATEUR": "",
        },
        {
            "COLONNE": PARTS_FAMILLES[famille],
            "DESCRIPTION": f"Part de la famille {famille}",
            "UNITE": "%",
            "DENOMINATEUR": COLONNE_TOTAL,
        },
        {
            "COLONNE": DENSITES[colonne_nombre],
            "DESCRIPTION": f"Densité de la famille {famille}",
            "UNITE": "observations pour 10 000 habitants",
            "DENOMINATEUR": COLONNE_POPULATION,
        },
    ])

lignes_dictionnaire.extend([
    {
        "COLONNE": COLONNE_TOTAL,
        "DESCRIPTION": "Total des observations du périmètre J10c",
        "UNITE": "observations",
        "DENOMINATEUR": "",
    },
    {
        "COLONNE": DENSITES[COLONNE_TOTAL],
        "DESCRIPTION": "Densité du périmètre J10c",
        "UNITE": "observations pour 10 000 habitants",
        "DENOMINATEUR": COLONNE_POPULATION,
    },
    {
        "COLONNE": "NB_TYPES_SERVICES_PRESENTS_J10C",
        "DESCRIPTION": "Nombre de types BPE distincts présents",
        "UNITE": "types",
        "DENOMINATEUR": "",
    },
])

dictionnaire = pd.DataFrame(lignes_dictionnaire)

assert set(dictionnaire["COLONNE"]) == (
    set(indicateurs.columns) - {"CODGEO"}
)

sauvegarder_csv(
    dictionnaire,
    DOSSIER_INTERIM / "dictionnaire_indicateurs_j10c.csv",
)

In [40]:
#Contrôles finaux

colonnes_effectifs = [
    *COLONNES_TYPES.values(),
    *COLONNES_FAMILLES.values(),
    COLONNE_TOTAL,
    "NB_TYPES_SERVICES_PRESENTS_J10C",
]

assert len(profil_j10c) == len(profil)
assert profil_j10c["CODGEO"].is_unique
assert set(profil_j10c["CODGEO"]) == codes_profil

assert services["ID_OBSERVATION"].is_unique
assert services["CODGEO"].isin(codes_profil).all()
assert not services["TYPEQU_ANALYSE"].eq("A504").any()

assert indicateurs[colonnes_effectifs].notna().all().all()
assert indicateurs[colonnes_effectifs].ge(0).all().all()

assert (
    indicateurs[colonnes_types].sum(axis=1)
    == indicateurs[COLONNE_TOTAL]
).all()

assert indicateurs[COLONNE_TOTAL].sum() == len(services)

for colonne in DENSITES.values():
    valeurs = indicateurs[colonne].dropna().astype(float)

    assert np.isfinite(valeurs).all()
    assert valeurs.ge(0).all()

    assert indicateurs.loc[
        ~population_valide, colonne
    ].isna().all()

pd.testing.assert_frame_equal(
    profil.sort_values("CODGEO").reset_index(drop=True),
    profil_j10c[profil.columns]
    .sort_values("CODGEO")
    .reset_index(drop=True),
)

controle = pd.DataFrame([{
    "COMMUNES_ENTREE": len(profil),
    "COMMUNES_SORTIE": len(profil_j10c),
    "OBSERVATIONS_SERVICES": len(services),
    "OBSERVATIONS_HORS_PROFIL": len(hors_profil),
    "OBSERVATIONS_SIRET_TYPE_REPETE": len(repetitions),
    "COMMUNES_SANS_POPULATION_VALIDE": int(
        (~population_valide).sum()
    ),
    "ETAT": (
        "CONTROLES_STRUCTURELS_OK"
        if repetitions.empty
        else "CONTROLES_STRUCTURELS_OK_REPETITIONS_A_EXAMINER"
    ),
}])

sauvegarder_csv(
    controle,
    DOSSIER_INTERIM / "controle_jointures.csv",
)

display(controle)

,COMMUNES_ENTREE,COMMUNES_SORTIE,OBSERVATIONS_SERVICES,OBSERVATIONS_HORS_PROFIL,OBSERVATIONS_SIRET_TYPE_REPETE,COMMUNES_SANS_POPULATION_VALIDE,ETAT
0,1266,1266,184760,0,0,0,CONTROLES_STRUCTURELS_OK


In [41]:
# Uniformiser les noms des colonnes J10c :
# NB_OBS_..._BPE2025
# PART_..._J10C_PCT
# DENSITE_OBS_..._10000_HAB

LIBELLES_GROUPES = {
    "A1": "SERVICES_PUBLICS",
    "A2": "BANQUES_POSTE_FUNERAIRE",
    "A3": "AUTOMOBILE_CONDUITE",
    "A4": "ARTISANS_BATIMENT",
    "A5": "SERVICES_PERSONNELS_IMMOBILIERS_EMPLOI",
}

# Reprendre les noms déjà définis dans le notebook.
LIBELLES_TYPES = NOMS_ANALYSE.copy()

# Remplacer aussi les noms techniques des services publics.
LIBELLES_TYPES.update({
    "A104": "GENDARMERIE",
    "A105": "COURS_APPEL",
    "A108": "PRUDHOMMES",
    "A109": "TRIBUNAUX_COMMERCE",
    "A120": "FINANCES_PUBLIQUES_REGIONALES",
    "A121": "FINANCES_PUBLIQUES_DEPARTEMENTALES",
    "A122": "FRANCE_TRAVAIL",
    "A124": "MAISONS_JUSTICE_DROIT",
    "A125": "ANTENNES_JUSTICE",
    "A126": "CONSEILS_ACCES_DROIT",
    "A128": "FRANCE_SERVICES",
    "A129": "MAIRIES",
    "A130": "BUREAUX_AIDE_JURIDICTIONNELLE",
    "A131": "TRIBUNAUX_JUDICIAIRES",
    "A132": "TRIBUNAUX_PROXIMITE",
    "A133": "DECHETERIES",
    "A134": "COURS_CRIMINELLES_DEPARTEMENTALES",
    "A135": "COURS_ASSISES",
    "A136": "JURIDICTIONS_ADMINISTRATIVES",
    "A137": "TRIBUNAUX_CORRECTIONNELS",
    "A138": "TRIBUNAUX_POLICE",
    "A139": "TRIBUNAUX_ENFANTS",
    "A140": "POLICE",
})

renommage = {}

# Sous-catégories
for code, libelle in LIBELLES_TYPES.items():
    renommage.update({
        f"NB_OBS_{code}_BPE2025":
            f"NB_OBS_{libelle}_BPE2025",

        f"PART_{code}_SERVICES_J10C_PCT":
            f"PART_{libelle}_J10C_PCT",

        f"DENSITE_OBS_{code}_10000_HAB":
            f"DENSITE_OBS_{libelle}_10000_HAB",
    })

# Groupes
for code, libelle in LIBELLES_GROUPES.items():
    renommage.update({
        f"NB_OBS_SERVICES_{code}_BPE2025":
            f"NB_OBS_{libelle}_BPE2025",

        f"PART_{code}_SERVICES_J10C_PCT":
            f"PART_{libelle}_J10C_PCT",

        f"DENSITE_OBS_SERVICES_{code}_10000_HAB":
            f"DENSITE_OBS_{libelle}_10000_HAB",
    })

# Vérifier les futurs noms avant toute modification.
for nom_table, table in [
    ("profil_j10c", profil_j10c),
    ("indicateurs", indicateurs),
]:
    futurs_noms = [
        renommage.get(colonne, colonne)
        for colonne in table.columns
    ]

    if len(futurs_noms) != len(set(futurs_noms)):
        raise ValueError(
            f"Le renommage créerait des doublons dans {nom_table}."
        )

# Renommer : les valeurs restent identiques.
profil_j10c = profil_j10c.rename(columns=renommage)
indicateurs = indicateurs.rename(columns=renommage)

# Actualiser le dictionnaire des indicateurs.
dictionnaire["COLONNE"] = (
    dictionnaire["COLONNE"].replace(renommage)
)

assert set(dictionnaire["COLONNE"]) == (
    set(indicateurs.columns) - {"CODGEO"}
)

# Conserver une correspondance entre anciens et nouveaux noms.
correspondance_colonnes = pd.DataFrame(
    renommage.items(),
    columns=["ANCIEN_NOM", "NOUVEAU_NOM"],
)

# Réenregistrer les fichiers.
sauvegarder_csv(profil_j10c, FICHIER_SORTIE)
sauvegarder_csv(indicateurs, FICHIER_COMMUNES)

sauvegarder_csv(
    dictionnaire,
    DOSSIER_INTERIM / "dictionnaire_indicateurs_j10c.csv",
)

sauvegarder_csv(
    correspondance_colonnes,
    DOSSIER_INTERIM / "correspondance_colonnes_j10c.csv",
)

print("Colonnes normalisées et fichiers réenregistrés.")
print("Profil :", FICHIER_SORTIE)

display(correspondance_colonnes.head(15))

Colonnes normalisées et fichiers réenregistrés.
Profil : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10c.csv


,ANCIEN_NOM,NOUVEAU_NOM
0,NB_OBS_A104_BPE2025,NB_OBS_GENDARMERIE_BPE2025
1,PART_A104_SERVICES_J10C_PCT,PART_GENDARMERIE_J10C_PCT
2,DENSITE_OBS_A104_10000_HAB,DENSITE_OBS_GENDARMERIE_10000_HAB
3,NB_OBS_A105_BPE2025,NB_OBS_COURS_APPEL_BPE2025
4,PART_A105_SERVICES_J10C_PCT,PART_COURS_APPEL_J10C_PCT
5,DENSITE_OBS_A105_10000_HAB,DENSITE_OBS_COURS_APPEL_10000_HAB
6,NB_OBS_A108_BPE2025,NB_OBS_PRUDHOMMES_BPE2025
7,PART_A108_SERVICES_J10C_PCT,PART_PRUDHOMMES_J10C_PCT
8,DENSITE_OBS_A108_10000_HAB,DENSITE_OBS_PRUDHOMMES_10000_HAB
9,NB_OBS_A109_BPE2025,NB_OBS_TRIBUNAUX_COMMERCE_BPE2025


In [42]:
profil_j10c

,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU,DENSITE_RESTAURANTS_10000_HAB,DENSITE_RESTAURATION_RAPIDE_10000_HAB,DENSITE_RESTAURATION_TRAD_10000_HAB,PART_RESTAURATION_RAPIDE_PCT,PART_RESTAURATION_TRADITIONNELLE_PCT,INDICE_DENSITE_RAPIDE_IDF_BASE100,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,...,DENSITE_OBS_GENDARMERIE_10000_HAB,DENSITE_OBS_COURS_APPEL_10000_HAB,DENSITE_OBS_PRUDHOMMES_10000_HAB,DENSITE_OBS_TRIBUNAUX_COMMERCE_10000_HAB,DENSITE_OBS_FINANCES_PUBLIQUES_REGIONALES_10000_HAB,DENSITE_OBS_FINANCES_PUBLIQUES_DEPARTEMENTALES_10000_HAB,DENSITE_OBS_FRANCE_TRAVAIL_10000_HAB,DENSITE_OBS_MAISONS_JUSTICE_DROIT_10000_HAB,DENSITE_OBS_ANTENNES_JUSTICE_10000_HAB,DENSITE_OBS_CONSEILS_ACCES_DROIT_10000_HAB,DENSITE_OBS_FRANCE_SERVICES_10000_HAB,DENSITE_OBS_MAIRIES_10000_HAB,DENSITE_OBS_BUREAUX_AIDE_JURIDICTIONNELLE_10000_HAB,DENSITE_OBS_TRIBUNAUX_JUDICIAIRES_10000_HAB,DENSITE_OBS_TRIBUNAUX_PROXIMITE_10000_HAB,DENSITE_OBS_DECHETERIES_10000_HAB,DENSITE_OBS_COURS_CRIMINELLES_DEPARTEMENTALES_10000_HAB,DENSITE_OBS_COURS_ASSISES_10000_HAB,DENSITE_OBS_JURIDICTIONS_ADMINISTRATIVES_10000_HAB,DENSITE_OBS_TRIBUNAUX_CORRECTIONNELS_10000_HAB,DENSITE_OBS_TRIBUNAUX_POLICE_10000_HAB,DENSITE_OBS_TRIBUNAUX_ENFANTS_10000_HAB,DENSITE_OBS_POLICE_10000_HAB,DENSITE_OBS_BANQUES_10000_HAB,DENSITE_OBS_FUNERAIRE_10000_HAB,DENSITE_OBS_POSTE_BUREAUX_10000_HAB,DENSITE_OBS_POSTE_RELAIS_10000_HAB,DENSITE_OBS_POSTE_AGENCES_10000_HAB,DENSITE_OBS_REPARATION_VEHICULES_MATERIEL_AGRICOLE_10000_HAB,DENSITE_OBS_CONTROLE_VEHICULES_10000_HAB,DENSITE_OBS_LOCATION_VEHICULES_10000_HAB,DENSITE_OBS_FORMATION_CONDUITE_10000_HAB,DENSITE_OBS_MACONNERIE_10000_HAB,DENSITE_OBS_PLATRERIE_PEINTURE_10000_HAB,DENSITE_OBS_MENUISERIE_CHARPENTE_SERRURERIE_10000_HAB,DENSITE_OBS_PLOMBERIE_COUVERTURE_CHAUFFAGE_10000_HAB,DENSITE_OBS_ELECTRICITE_10000_HAB,DENSITE_OBS_ENTREPRISES_BATIMENT_10000_HAB,DENSITE_OBS_COIFFURE_10000_HAB,DENSITE_OBS_VETERINAIRES_10000_HAB,DENSITE_OBS_INTERIM_10000_HAB,DENSITE_OBS_IMMOBILIER_10000_HAB,DENSITE_OBS_ENTRETIEN_LINGE_10000_HAB,DENSITE_OBS_BEAUTE_ONGLES_10000_HAB,DENSITE_OBS_SERVICES_PUBLICS_10000_HAB,DENSITE_OBS_BANQUES_POSTE_FUNERAIRE_10000_HAB,DENSITE_OBS_AUTOMOBILE_CONDUITE_10000_HAB,DENSITE_OBS_ARTISANS_BATIMENT_10000_HAB,DENSITE_OBS_SERVICES_PERSONNELS_IMMOBILIERS_EMPLOI_10000_HAB,DENSITE_OBS_SERVICES_HORS_RESTAURATION_10000_HAB
0,75056,Paris,75,11,2249975.0,2190327.0,2113705.0,274280.561508206,513386.75602884,457511.658563684,387099.215350194,302819.494109117,154040.332161959,24566.9822779995,787667.317537046,970898.414592524,481426.8085490755,178607.31443995849,12.98,24.29,21.65,37.26,45.93,22.78,8.45,-76622.0,-3.5,-0.59,33650.0,16.8,1551133.75022,188002.0,73.38459010221389,88.94429449710343,20911,8329,12553,29,0,98.93055085738077,39.40474191053151,59.38860910108081,39.8307111089857,60.03060590120033,200.61215141394868,54155.23178,51427.58154,87388.40827,75287.99165,59522.89891,...,0.004731,0.004731,0.004731,0.004731,0.108814,0.0,0.085159,0.014193,0.0,0.004731,0.056772,0.094621,0.004731,0.004731,0.0,0.047310,0.004731,0.004731,0.009462,0.004731,0.004731,0.004731,0.089890,5.984752,0.988785,0.605572,0.179779,0.000000,4.314699,0.123007,5.052739,1.636936,9.490445,15.938837,8.312418,13.909226,13.876109,16.847195,16.785691,1.

In [43]:
#Enregistrer les résultats

sauvegarder_csv(services, FICHIER_DETAIL)
sauvegarder_csv(indicateurs, FICHIER_COMMUNES)
sauvegarder_csv(profil_j10c, FICHIER_SORTIE)

metadonnees = {
    "notebook": "10c_services_particuliers_idf.ipynb",
    "date_traitement_utc": DATE_TRAITEMENT,
    "source": "INSEE_BPE_2025",
    "url_nomenclature": (
        "https://www.insee.fr/fr/metadonnees/source/fichier/"
        "BPE25_liste_hierarchisee_TYPEQU.html"
    ),
    "fichier_bpe": str(FICHIER_BPE),
    "sha256_bpe": HASH_BPE,
    "profil_entree": str(FICHIER_PROFIL),
    "sha256_profil_entree": sha256_fichier(FICHIER_PROFIL),
    "profil_sortie": str(FICHIER_SORTIE),
    "codes_retenus": CODES_RETENUS,
    "codes_exclus": ["A504"],
    "familles": FAMILLES,
    "colonne_population": COLONNE_POPULATION,
    "nombre_observations": len(services),
    "nombre_communes": len(profil_j10c),
    "observations_siret_type_repete": len(repetitions),
    "deduplication_automatique": False,
    "denominateur_parts": COLONNE_TOTAL,
    "limites": [
        "Comptage d'observations BPE, pas de locaux uniques.",
        "Ouverture actuelle non vérifiée.",
        "Accueil du public non vérifié.",
        "Familles de services hétérogènes.",
        "Aucune mesure de fréquentation.",
        "Contrôle des coordonnées limité à un rectangle géographique.",
    ],
}

with open(
    DOSSIER_INTERIM / "metadata_j10c.json",
    "w",
    encoding="utf-8",
) as flux:
    json.dump(
        metadonnees,
        flux,
        ensure_ascii=False,
        indent=2,
    )

print("Détail :", FICHIER_DETAIL)
print("Indicateurs communaux :", FICHIER_COMMUNES)
print("Profil enrichi :", FICHIER_SORTIE)
print("J10c enregistré.")

Détail : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\services_particuliers_idf_j10c.csv
Indicateurs communaux : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j10c\services_particuliers_communes.csv
Profil enrichi : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10c.csv
J10c enregistré.
